Es común que te encuentres en una auditoría con integraciones de saldos, por ejemplo en cuentas por cobrar o por pagar, donde por algunas circunstancias no se tienen identificadas (o integradas) las facturas que corresponden al pago, ejemplo: un pago a un proveedor puede ser de $10,000 pesos, ese importe puede estar pagando las facturas A, B y C o puede estar pagando las facturas D y E, en algunas situaciones la información puede estar deliberadamente escondida o modificada. El siguiente código identifica, partiendo de un número de cliente, las facturas que podrían integrar el importe del pago (o cobro). El código replica lo que Solver en Excel puede solucionar, pero de una forma más rápida y escalable. 

In [1]:
# pip install pulp

In [2]:
import pandas as pd
from scipy.optimize import linprog
from pulp import LpProblem, LpVariable, lpSum, value

In [3]:
# importar los datos
df = pd.read_excel(r'data\integrador_pagos.xlsx')
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,Factura,NumCliente,Cliente,Importe
1,NaN,FV0019173,10089,Gastrolab,23646.57
2,NaN,FV0019174,10090,Olivo Foods,11372
3,NaN,FV0019175,10056,Estralis,23983
4,NaN,FV0019176,10092,Lala 99,28930


en este caso se debe hacer una limpieza inicial, puesto que los headers están en la fila 0 y hay una columna llamada Unnamed:0 que no se utilizará.

In [4]:
# solamente especificando usecols y header se arregla la situación
df = pd.read_excel(r'data\integrador_pagos.xlsx', usecols='B:E', header=1)
df.head()

,Factura,NumCliente,Cliente,Importe
0,FV0019173,10089,Gastrolab,23646.57
1,FV0019174,10090,Olivo Foods,11372.00
2,FV0019175,10056,Estralis,23983.00
3,FV0019176,10092,Lala 99,28930.00
4,FV0019177,10090,Olivo Foods,17686.00


In [ ]:
# inspección del tipo de datos
df.dtypes

Factura           str
NumCliente      int64
Cliente           str
Importe       float64
dtype: object

In [ ]:
# en caso de que se necesite, convertir la columna 'Importe' a números (flotantes)
# df['Importe'] = df['Importe'].str.replace(',', '', regex=True).astype(float)

# filtrar las facturas del cliente 10089
objetivo = df[df['NumCliente'] == 10089]

# crear una lista de los importes de las facturas
factura_importes = objetivo['Importe'].tolist()

# crear un problema de programación lineal
problema = LpProblem("FacturasCliente10089", sense=1)

# variables binarias para representar si seleccionamos cada factura
facturas_seleccionadas = [LpVariable(f"Factura_{i}", cat='Binary') for i in range(len(factura_importes))]

# agregar la función objetivo: minimizar la diferencia
problema += lpSum([facturas_seleccionadas[i] * factura_importes[i] for i in range(len(factura_importes))])

# restricción: la suma de las facturas seleccionadas debe ser igual a 68,680.42
problema += lpSum([facturas_seleccionadas[i] * factura_importes[i] for i in range(len(factura_importes))]) == 68680.42

# resolver el problema de programación lineal
problema.solve()

# mostrar las facturas seleccionadas
facturas_seleccionadas_indices = [i for i, var in enumerate(facturas_seleccionadas) if value(var) == 1]
facturas_seleccionadas = objetivo.iloc[facturas_seleccionadas_indices]
facturas_seleccionadas

,Factura,NumCliente,Cliente,Importe
0,FV0019173,10089,Gastrolab,23646.57
12,FV0019185,10089,Gastrolab,24217.85
15,FV0019188,10089,Gastrolab,20816.00


### Función 'Integrador'

In [ ]:
def Integrador(cliente, pago, df):
    
    # convertir la columna 'Importe' a números (flotantes)
    #df['Importe'] = df['Importe'].str.replace(',', '', regex=True).astype(float)

    # filtrar las facturas del cliente especificado
    cliente_data = df[df['NumCliente'] == cliente]

    # crear una lista de los importes de las facturas
    factura_importes = cliente_data['Importe'].tolist()

    # crear un problema de programación lineal
    problema = LpProblem(f"FacturasCliente{cliente}", sense=1)

    # variables binarias para representar si seleccionamos cada factura
    facturas_seleccionadas = [LpVariable(f"Factura_{i}", cat='Binary') for i in range(len(factura_importes))]

    # agregar la función objetivo: minimizar la diferencia
    problema += lpSum([facturas_seleccionadas[i] * factura_importes[i] for i in range(len(factura_importes))])

    # restricción: la suma de las facturas seleccionadas debe ser igual al pago especificado
    problema += lpSum([facturas_seleccionadas[i] * factura_importes[i] for i in range(len(factura_importes))]) == pago

    # resolver el problema de programación lineal
    problema.solve()

    # mostrar las facturas seleccionadas
    facturas_seleccionadas_indices = [i for i, var in enumerate(facturas_seleccionadas) if value(var) == 1]
    facturas_seleccionadas = cliente_data.iloc[facturas_seleccionadas_indices]

    return facturas_seleccionadas

In [ ]:
# prueba para el cliente número 10089 con un pago de $68,680.42
Integrador(10089, 68680.42, df)

,Factura,NumCliente,Cliente,Importe
0,FV0019173,10089,Gastrolab,23646.57
12,FV0019185,10089,Gastrolab,24217.85
15,FV0019188,10089,Gastrolab,20816.00


### Conclusión

La función creada implementa un modelo de programación lineal para realizar la conciliación de pagos contra facturas de un cliente específico. El objetivo es identificar qué combinación de facturas suma exactamente el importe de un pago recibido, seleccionando mediante variables binarias aquellas facturas que deben considerarse liquidadas. En resumen, esta implementación constituye una herramienta efectiva para la identificación automática y exacta de facturas asociadas a un pago, facilitando los procesos de conciliación en casos donde hay información parcial o desaparecida.